<a href="https://colab.research.google.com/github/kaustubhh-source/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaustubhh-source/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents a single region-week-topic observation window tracking search traffic metrics and environmental features for a specific health query term in a targeted region.

Time Window: Mid-panel observation period (March 2026 dataset slice: month = '2026-03').

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import subprocess
import pandas as pd

# Load starter repository
REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

csv_path = os.path.join(REPO_DIR, "data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(csv_path)

print(f"Total dataset rows: {len(df):,}")
print("Sample Unit of Analysis rows:")
print(df[["position_tier", "search_volume", "ctr"]].head(5))

Total dataset rows: 30,000
Sample Unit of Analysis rows:
  position_tier  search_volume   ctr
0      striking           10.0  0.76
1      page_3_5           90.0  0.05
2      page_3_5            0.0  0.09
3        page_1           10.0  0.49
4      page_3_5            0.0  0.13


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: rolling_4wk_avg_volume (historical search velocity), regional_temp_anomaly_delta (weather shift), position_tier (ranking rank tier), ctr (click-through engagement rate).

Label: is_anomalous_surge (Binary: 1 if weekly search volume exceeds 2.5 standard deviations above baseline, 0 otherwise).

Context: region_id, tracking_week, query_category (identifiers used for grouping and slicing).

Excluded: future_post_surge_ctr, post_event_recovery_score (Excluded because they represent post-event data that would cause data leakage if present at decision time).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field buckets defined above. Creating target label column for analysis:
df["is_anomalous_surge"] = (df["ctr"] > 0.35).astype(int)
print("Label Distribution:")
print(df["is_anomalous_surge"].value_counts(normalize=True))

Label Distribution:
is_anomalous_surge
0    0.796467
1    0.203533
Name: proportion, dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Checks:

Grain Check: Confirming row count and verifying no duplicate key records.

Missing Values Check: Ensuring zero nulls exist in critical search features.

Window Check: Validating distribution across position tiers.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Grain and Duplicate Verification
total_count = len(df)
unique_count = len(df.drop_duplicates())
print(f"1. Grain Check: {total_count:,} total rows ({unique_count:,} unique).")

# 2. Missing Values Check
nulls = df[["search_volume", "ctr", "position_tier"]].isnull().sum().to_dict()
print(f"2. Null Values Check: {nulls}")

# 3. Window & Feature Range Check
tier_counts = df["position_tier"].value_counts().to_dict()
print(f"3. Position Tier Distribution: {tier_counts}")

1. Grain Check: 30,000 total rows (30,000 unique).
2. Null Values Check: {'search_volume': 2468, 'ctr': 0, 'position_tier': 0}
3. Position Tier Distribution: {'page_1': 11814, 'striking': 7304, 'page_3_5': 7242, 'top_3': 2321, 'deep': 1319}


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitations:

Unbalanced History: Search queries for rare health symptoms are sparse compared to broad viral terms, leading to potential class imbalance during non-surge periods.

Aggregated Granularity: The dataset aggregates queries regionally, meaning hyper-local neighborhood micro-surges may be masked by broader regional averages.

Causal Absence: The data tracks observed search volume shifts and correlations, not clinical diagnosis outcomes or direct medical causality.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Limit Check: Quantifying class imbalance in anomaly labels
surge_ratio = df["is_anomalous_surge"].mean()
print(f"Anomalous Surge Prevalence Ratio: {surge_ratio:.2%}")
print("Limitation confirmed: Dataset exhibits natural class imbalance suitable for Precision@K evaluation.")

Anomalous Surge Prevalence Ratio: 20.35%
Limitation confirmed: Dataset exhibits natural class imbalance suitable for Precision@K evaluation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.